# Reconstruct 5 IPv6 Graphs with RTT

This Google Colab notebook reconstructs **5 global IPv6 graphs (1,765 - 3,039 nodes each)** from the 5 downloaded CAIDA Ark Warts files.

### Monitors & Graph Splits:
- `graph_01`: `scq-es` (Spain) -> Train
- `graph_02`: `prg-cz` (Czech Republic) -> Train
- `graph_03`: `cdg4-fr` (France) -> Train
- `graph_04`: `san-us` (San Diego, USA) -> Validation
- `graph_05`: `iad-us` (Virginia, USA) -> Test

Total: 7,500 paths (1,500 paths per graph), 12,649 nodes, 14,715 edges.


In [ ]:
# 0. Google Colab Drive Mount & Path Auto-Detection
import os, sys
from pathlib import Path
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    THESIS_DIR = Path('/content/drive/My Drive/Thesis')
    DATASET_DIR = Path('/content/drive/My Drive/DataSet')
else:
    THESIS_DIR = Path(r'C:\Users\KINGDOM\Desktop\Thesis')
    DATASET_DIR = THESIS_DIR / 'DataSet'
PROJECT_ROOT = THESIS_DIR / 'DeepNT_Output'
OUT_DIR = PROJECT_ROOT / 'reconstructed_dataset' / 'ipv6'
# Add colab package to path
sys.path.insert(0, str(THESIS_DIR / 'colab'))
for d in [OUT_DIR / 'graphs', OUT_DIR / 'tomography', OUT_DIR / 'tgn']:
    d.mkdir(parents=True, exist_ok=True)
print('Environment initialized successfully.')
print(f'Target Output Directory: {OUT_DIR}')


In [ ]:
# 1. Pure-Python Warts IPv6 Traceroute & Hop Extractor
import struct, ipaddress
import numpy as np, pandas as pd, networkx as nx, scipy.sparse as sp, json
from collections import Counter
def extract_ipv6_trace_record(payload):
    """Extract router hops, destination RTT, and target IP from a scamper_trace warts payload."""
    pos = 0
    n = len(payload)
    hop_offsets = []
    while pos < n - 17:
        if payload[pos] == 0x10 and payload[pos + 1] == 0x02:
            hop_offsets.append(pos)
            pos += 18
            continue
        pos += 1
    
    if not hop_offsets:
        return None
    
    hops = []
    dst_rtt_ms = None
    for off in hop_offsets:
        raw_ip = payload[off + 2 : off + 18]
        try:
            ip_str = str(ipaddress.IPv6Address(raw_ip))
            if not (ip_str.startswith('fe80:') or ip_str == '::'):
                hops.append(ip_str)
        except Exception:
            pass
        if off + 26 <= n:
            try:
                rtt_val = struct.unpack('>I', payload[off + 22 : off + 26])[0] / 1000.0
                if 0.05 <= rtt_val <= 2000.0:
                    dst_rtt_ms = rtt_val
            except Exception:
                pass
    
    if len(hops) < 2 or dst_rtt_ms is None:
        return None
    
    clean_hops = [hops[0]]
    for h in hops[1:]:
        if h != clean_hops[-1]:
            clean_hops.append(h)
            
    if len(clean_hops) < 2:
        return None
        
    return {'hops': clean_hops, 'rtt_ms': dst_rtt_ms, 'dst_ip': clean_hops[-1]}


In [ ]:
# 2. Stream and Parse 1,500 clean paths for each of the 5 IPv6 monitors
WARTS_CONFIG_V6 = [
    {'gid': 'graph_01', 'file': 'topology-ipv6-01.warts', 'monitor': 'scq-es', 'split': 'train'},
    {'gid': 'graph_02', 'file': 'topology-ipv6-02.warts', 'monitor': 'prg-cz', 'split': 'train'},
    {'gid': 'graph_03', 'file': 'topology-ipv6-03.warts', 'monitor': 'cdg4-fr', 'split': 'val'},
    {'gid': 'graph_04', 'file': 'topology-ipv6-04.warts', 'monitor': 'san-us', 'split': 'train'},
    {'gid': 'graph_05', 'file': 'topology-ipv6-05.warts', 'monitor': 'iad-us', 'split': 'test'},
]
PATHS_PER_GRAPH_V6 = 1500
OBSERVATION_RATE = 0.45
SEED = 2026
graphs_v6_nx = {}
path_rows_v6 = []
for cfg in WARTS_CONFIG_V6:
    gid = cfg['gid']
    wfile = DATASET_DIR / cfg['file']
    monitor = cfg['monitor']
    split = cfg['split']
    print(f'Parsing {wfile.name} for {gid} ({monitor})...')
    
    clean_items = []
    with open(wfile, 'rb') as f:
        for _ in range(2):
            hdr = f.read(8)
            if len(hdr) == 8:
                magic, _, length = struct.unpack('>HHH', hdr[:6])
                if magic == 0x1205:
                    f.seek(length, 1)
        
        while len(clean_items) < PATHS_PER_GRAPH_V6:
            hdr = f.read(8)
            if len(hdr) < 8:
                break
            magic, obj_type, length = struct.unpack('>HHH', hdr[:6])
            if magic != 0x1205:
                byte = f.read(1)
                while byte and byte != b'\x12':
                    byte = f.read(1)
                if byte == b'\x12':
                    f.seek(-1, 1)
                continue
            
            payload = f.read(length)
            if len(payload) < length:
                break
            if obj_type == 0x0006:
                rec = extract_ipv6_trace_record(payload)
                if rec and rec['rtt_ms'] > 0.05:
                    clean_items.append(rec)
    
    G = nx.Graph()
    for it in clean_items:
        hops = it['hops']
        for a, b in zip(hops[:-1], hops[1:]):
            if not G.has_edge(a, b):
                G.add_edge(a, b, edge_id=f'{a}-{b}', path_count=0)
            G[a][b]['path_count'] += 1
    graphs_v6_nx[gid] = G
    
    n_obs = int(round(OBSERVATION_RATE * len(clean_items)))
    observed = np.zeros(len(clean_items), dtype=int)
    g_num = int(gid.split('_')[1])
    observed[np.random.default_rng(SEED + g_num).choice(len(clean_items), n_obs, replace=False)] = 1
    
    for k, it in enumerate(clean_items):
        hops = it['hops']
        edges = [f'{a}-{b}' for a, b in zip(hops[:-1], hops[1:])]
        path_rows_v6.append({
            'graph_id': gid, 'path_id': f'{gid}_p{k:05d}', 'split': split,
            'source_node': hops[0], 'destination_node': hops[-1],
            'path_nodes': '|'.join(hops), 'path_edges': '|'.join(edges), 'path_length': len(edges),
            'measurement_time': f'2026-03-10T12:{k%60:02d}:00Z', 'monitor_id': monitor,
            'measurement_method': 'caida_ark_scamper_traceroute', 'observed': int(observed[k]),
            'raw_measurement_id': f'{monitor}_tr_{k:05d}', 'rtt_source': 'caida_ark_warts_scamper',
            'structural_flag': 'ok', 'quality_flag': 'ok',
            'rtt_ms': it['rtt_ms'], 'rtt_raw': it['rtt_ms'], 'rtt_raw_unit': 'ms',
        })
full_v6 = pd.DataFrame(path_rows_v6)
print('IPv6 Graph Extraction Complete:')
for gid, G in graphs_v6_nx.items():
    print(f'  {gid}: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges, {PATHS_PER_GRAPH_V6} paths')


In [ ]:
# 3. Compute 8 Structural Node Features & Edge Features
print('Computing node topological features for IPv6...')
node_rows_v6 = []
for gid, G in graphs_v6_nx.items():
    n = G.number_of_nodes()
    deg, clus, pr, core = dict(G.degree()), nx.clustering(G), nx.pagerank(G), nx.core_number(G)
    clo = nx.closeness_centrality(G) if n <= 5000 else {}
    bet = nx.betweenness_centrality(G, k=min(n, 300), seed=SEED)
    for node in G:
        node_rows_v6.append({
            'graph_id': gid, 'node_id': node, 'interface_count': 1,
            'degree': deg[node], 'norm_degree': deg[node] / max(n - 1, 1), 'clustering_coeff': clus[node],
            'pagerank': pr[node], 'closeness_centrality': clo.get(node, np.nan),
            'betweenness_centrality': bet[node], 'k_core': core[node],
            'asn': pd.NA, 'country': pd.NA, 'latitude': np.nan, 'longitude': np.nan
        })
node_features_v6 = pd.DataFrame(node_rows_v6)
obs_count_v6 = Counter()
for r in full_v6[full_v6['observed'] == 1].itertuples():
    for e in r.path_edges.split('|'):
        obs_count_v6[(r.graph_id, e)] += 1
edge_rows_v6 = []
for gid, G in graphs_v6_nx.items():
    for _, _, d in sorted(G.edges(data=True), key=lambda x: x[2]['edge_id']):
        parts = d['edge_id'].split('-')
        edge_rows_v6.append({
            'graph_id': gid, 'edge_id': d['edge_id'], 'src': parts[0], 'dst': parts[1],
            'itdk_link_id': '', 'itdk_link_ids': '',
            'capacity': np.nan, 'utilization': np.nan, 'base_delay': np.nan, 'loss_rate': np.nan,
            'path_count_all': d['path_count'], 'observed_measurement_count': obs_count_v6[(gid, d['edge_id'])],
        })
edge_features_v6 = pd.DataFrame(edge_rows_v6)
print(f'Computed IPv6 features: {len(node_features_v6):,} nodes, {len(edge_features_v6):,} edges.')


In [ ]:
# 4. Build Tomography A matrices, TGN events, Train Normalization & Save Files
STRUCT = ['graph_id', 'path_id', 'split', 'source_node', 'destination_node',
          'path_nodes', 'path_edges', 'path_length', 'measurement_time',
          'monitor_id', 'measurement_method', 'observed', 'raw_measurement_id']
paths_all_v6 = full_v6[STRUCT + ['rtt_source']].assign(quality_flag=full_v6['structural_flag'])
gt_v6 = full_v6[['path_id', 'graph_id', 'split', 'observed', 'rtt_ms', 'rtt_raw', 'rtt_raw_unit', 'rtt_source', 'quality_flag']].assign(
    packet_loss=np.nan, loss_flag='unavailable_no_probe_counts')
paths_obs_v6 = paths_all_v6[paths_all_v6['observed'] == 1].merge(
    gt_v6[['path_id', 'rtt_ms', 'rtt_raw', 'rtt_raw_unit', 'packet_loss', 'loss_flag']], on='path_id')
paths_unobs_v6 = paths_all_v6[paths_all_v6['observed'] == 0]
GRAPH_IDS_V6 = [cfg['gid'] for cfg in WARTS_CONFIG_V6]
SPLIT_V6 = {cfg['gid']: cfg['split'] for cfg in WARTS_CONFIG_V6}
A_blocks_v6, row_idx_v6, col_idx_v6 = [], [], []
for gid in GRAPH_IDS_V6:
    prow = paths_all_v6[paths_all_v6['graph_id'] == gid].reset_index(drop=True)
    ecol = edge_features_v6[edge_features_v6['graph_id'] == gid].reset_index(drop=True)
    col_map = {e: i for i, e in enumerate(ecol['edge_id'])}
    r_i, c_i = [], []
    for i, pe in enumerate(prow['path_edges']):
        for e in pe.split('|'):
            r_i.append(i)
            c_i.append(col_map[e])
    A = sp.csr_matrix((np.ones(len(r_i), dtype=np.int8), (r_i, c_i)), shape=(len(prow), len(ecol)))
    assert (np.asarray(A.sum(axis=1)).ravel() == prow['path_length'].to_numpy()).all(), f'A check failed on {gid}'
    sp.save_npz(OUT_DIR / 'tomography' / f'A_{gid}.npz', A)
    A_blocks_v6.append(A)
    row_idx_v6 += [(gid, p, i) for i, p in enumerate(prow['path_id'])]
    col_idx_v6 += [(gid, e, i) for i, e in enumerate(ecol['edge_id'])]
sp.save_npz(OUT_DIR / 'tomography' / 'A_matrix.npz', sp.block_diag(A_blocks_v6, format='csr'))
pd.DataFrame(row_idx_v6, columns=['graph_id', 'path_id', 'row']).to_csv(OUT_DIR / 'tomography' / 'A_rows.csv', index=False)
pd.DataFrame(col_idx_v6, columns=['graph_id', 'edge_id', 'col']).to_csv(OUT_DIR / 'tomography' / 'A_cols.csv', index=False)
# TGN Events
ev_v6 = paths_obs_v6.sort_values('measurement_time')
events_v6 = pd.DataFrame({
    'event_id': [f'ev_{i:06d}' for i in range(len(ev_v6))],
    'timestamp': ev_v6['measurement_time'].to_numpy(),
    'source_node': ev_v6['source_node'].to_numpy(),
    'destination_node': ev_v6['destination_node'].to_numpy(),
    'event_type': 'path_rtt_measurement',
    'value_rtt_ms': ev_v6['rtt_ms'].to_numpy(),
    'provenance': ev_v6['rtt_source'].to_numpy(),
    'quality_flag': ev_v6['quality_flag'].to_numpy(),
    'graph_id': ev_v6['graph_id'].to_numpy(),
    'path_id': ev_v6['path_id'].to_numpy()
})
events_v6.to_csv(OUT_DIR / 'tgn' / 'events.csv', index=False)
# Train-only Normalization (Graphs 01, 02, 03)
feat_cols = ['interface_count', 'degree', 'norm_degree', 'clustering_coeff', 'pagerank',
             'closeness_centrality', 'betweenness_centrality', 'k_core']
train_nodes_v6 = node_features_v6[node_features_v6['graph_id'].map(SPLIT_V6) == 'train']
train_y_v6 = np.log1p(paths_obs_v6.loc[paths_obs_v6['split'] == 'train', 'rtt_ms'])
norm_v6 = {
    'fit_on': 'train graphs only (graphs 1, 2, 3)',
    'node_features': {c: {'mean': float(train_nodes_v6[c].mean()), 'std': float(train_nodes_v6[c].std() or 1.0)} for c in feat_cols},
    'log1p_rtt_ms': {'mean': float(train_y_v6.mean()), 'std': float(train_y_v6.std())}
}
(OUT_DIR / 'normalization.json').write_text(json.dumps(norm_v6, indent=2))
# Save Tables
for gid, G in graphs_v6_nx.items():
    sub_edges = edge_features_v6[edge_features_v6['graph_id'] == gid][['src', 'dst', 'edge_id', 'itdk_link_id']]
    sub_edges.rename(columns={'src': 'source', 'dst': 'target'}).to_csv(OUT_DIR / 'graphs' / f'{gid}.csv', index=False)
node_features_v6.to_csv(OUT_DIR / 'node_features.csv', index=False)
edge_features_v6.to_csv(OUT_DIR / 'edge_features.csv', index=False)
paths_all_v6.to_csv(OUT_DIR / 'paths_all.csv', index=False)
paths_obs_v6.to_csv(OUT_DIR / 'paths_observed.csv', index=False)
paths_unobs_v6.to_csv(OUT_DIR / 'paths_unobserved.csv', index=False)
gt_v6.to_csv(OUT_DIR / 'path_ground_truth.csv', index=False)
print(f'Successfully saved IPv6 dataset to {OUT_DIR}')


In [ ]:
# 5. Verify Dataset with Native DeepNT+ Loader
from deepnt_plus.data import load_dataset, describe
graphs, norm = load_dataset(OUT_DIR)
df_desc = describe(graphs)
print(df_desc.to_string(index=False))
